In [ ]:
# Upload datasets as .csv files in your colab session, and then run this cell.
# Dataset Link: https://drive.google.com/drive/folders/1PzURxyRaN5rclRcyRSI9G8oa1XVSo_SF?usp=drive_link

# Don't work in this colab. Make a copy in your drive instead.

import pandas as pd

copd_df = '/content/COPD_data_preprocessed.csv'
alzheimers_df = '/content/alzheimers_disease_data_preprocessed.csv'
arrhythmia_df = '/content/arrhythmia_data_preprocessed.csv'
autism_df = '/content/autism_screening_data_preprocessed.csv'
diabetes_df = '/content/diabetes_data_preprocessed.csv'
stroke_df = '/content/healthcare-dataset-stroke-data_preprocessed.csv'
heart_df = '/content/heart_disease_uci_preprocessed.csv'
hepatitis_df = '/content/hepatitis_data_preprocessed.csv'
liver_df = '/content/liver_patient_dataset_preprocessed.csv'
thyroid_df = '/content/thyroid_dataset_preprocessed.csv'

In [ ]:
# Cell 2: Disable hf_xet, pre-download TabPFN v2.0 checkpoint once, & verify GPU
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

!pip install -q --upgrade pip
!pip install -q tabpfn statsmodels scikit-learn scipy gdown tqdm

import gc
import glob
import shutil
import inspect
import warnings
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from tabpfn import TabPFNClassifier
from tabpfn.constants import ModelVersion

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch: {torch.__version__} | Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: Running on CPU! Switch to Runtime -> Change runtime type -> T4 GPU for 10x-30x faster execution.")

# Download TabPFN v2.0 checkpoint ONCE via plain HTTPS (bypasses hf_xet completely)
LOCAL_CKPT_PATH = "/content/tabpfn-v2-classifier-finetuned-zk73skhh.ckpt"
if not os.path.exists(LOCAL_CKPT_PATH) or os.path.getsize(LOCAL_CKPT_PATH) < 20_000_000:
    print("Downloading TabPFN v2.0 checkpoint via direct HTTPS...")
    !curl -sL "https://huggingface.co/Prior-Labs/TabPFN-v2-clf/resolve/main/tabpfn-v2-classifier-finetuned-zk73skhh.ckpt" -o "{LOCAL_CKPT_PATH}"

assert os.path.exists(LOCAL_CKPT_PATH) and os.path.getsize(LOCAL_CKPT_PATH) > 20_000_000, "Checkpoint download failed!"
print(f"TabPFN v2.0 checkpoint ready locally ({os.path.getsize(LOCAL_CKPT_PATH)/1e6:.1f} MB).")

# Lock HF Hub offline so the 50-fold loop NEVER touches the internet again
os.environ["HF_HUB_OFFLINE"] = "1"

_tabpfn_init_params = inspect.signature(TabPFNClassifier.__init__).parameters

PyTorch: 2.14.1+cpu | Device: cpu
TabPFN v2.0 checkpoint ready locally (29.0 MB).


In [ ]:
# Cell 3: Load the 10 preprocessed datasets from Cell 1 variables & verify schema
DATASET_CONFIGS = [
    {"Dataset": "COPD",         "Path": copd_df,       "Domain": "Pulmonology / Respiratory Medicine"},
    {"Dataset": "Alzheimers",   "Path": alzheimers_df, "Domain": "Neurology / Neurodegenerative Disease"},
    {"Dataset": "Arrhythmia",   "Path": arrhythmia_df, "Domain": "Cardiology"},
    {"Dataset": "Autism",       "Path": autism_df,     "Domain": "Neurodevelopmental / Psychiatry"},
    {"Dataset": "Diabetes",     "Path": diabetes_df,   "Domain": "Endocrinology / Metabolic Disease"},
    {"Dataset": "Stroke",       "Path": stroke_df,     "Domain": "Neurology / Cerebrovascular Disease"},
    {"Dataset": "Heart",        "Path": heart_df,      "Domain": "Cardiology"},
    {"Dataset": "Hepatitis",    "Path": hepatitis_df,  "Domain": "Hepatology / Infectious Disease"},
    {"Dataset": "Liver",        "Path": liver_df,      "Domain": "Hepatology / Gastroenterology"},
    {"Dataset": "Thyroid",      "Path": thyroid_df,    "Domain": "Endocrinology"},
]

# Auto-download from shared Google Drive folder if any CSV is missing in /content/
missing = [cfg["Path"] for cfg in DATASET_CONFIGS if not os.path.exists(cfg["Path"])]
if missing:
    print(f"Downloading {len(missing)} missing dataset(s) from Google Drive...")
    os.makedirs("/content/drive_download_tmp", exist_ok=True)
    !gdown --quiet --folder "https://drive.google.com/drive/folders/1PzURxyRaN5rclRcyRSI9G8oa1XVSo_SF" -O /content/drive_download_tmp
    for csv_path in glob.glob("/content/drive_download_tmp/**/*.csv", recursive=True):
        dest = os.path.join("/content", os.path.basename(csv_path))
        if not os.path.exists(dest):
            shutil.copy(csv_path, dest)

LOADED_DATASETS = {}
audit_rows = []

for cfg in DATASET_CONFIGS:
    ds_name = cfg["Dataset"]
    path = cfg["Path"]
    if not os.path.exists(path):
        raise FileNotFoundError(f"Dataset file not found: {path}")

    df = pd.read_csv(path)
    sex_col = df.columns[0]
    target_col = df.columns[-1]

    # Validate binary {0, 1} conventions (0=Negative/Female, 1=Positive/Male)
    df[target_col] = df[target_col].astype(int)
    assert set(df[target_col].unique()) == {0, 1}, f"{ds_name}: outcome must be binary {{0, 1}}"
    sex_vals = set(df[sex_col].dropna().astype(int).unique())
    assert sex_vals == {0, 1}, f"{ds_name}: sex column '{sex_col}' must be binary {{0, 1}}, found {sex_vals}"

    LOADED_DATASETS[ds_name] = df
    audit_rows.append({
        "Dataset": ds_name,
        "Domain": cfg["Domain"],
        "Rows": len(df),
        "Num_Feats": df.shape[1] - 1,
        "Sex_Col": sex_col,
        "Outcome_Col": target_col,
        "Missing_Values": int(df.isna().sum().sum()),
    })

display(pd.DataFrame(audit_rows))

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py", line 787, in urlopen
    response = self._make_request(
        conn,
    ...<10 lines>...
        **response_kw,
    )
  File "/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py", line 534, in _make_request
    response = conn.getresponse()
  File "/usr/local/lib/python3.13/dist-packages/urllib3/connection.py", line 565, in getresponse
    httplib_response = super().getresponse()
  File "/usr/lib/python3.13/http/client.py", line 1478, in getresponse
    response.begin()
    ~~~~~~~~~~~~~~^^
  File "/usr/lib/python3.13/http/client.py", line 343, in begin
    version, status, reason = self._read_status()
                              ~~~~~~~~~~~~~~~~~^^
  File "/usr/lib/python3.13/http/client.py", line 304, in _read_status
    line = str(self.fp.readline(_MAXLINE + 1), "iso-8859-1")
               ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/socket.

FileNotFoundError: Dataset file not found: /content/COPD_data_preprocessed.csv

In [ ]:
# Cell 4: Exact implementation of all 20 benchmark metrics (Fairness, Discrimination, Calibration, Robustness)
import statsmodels.api as sm
from statsmodels.tools.sm_exceptions import PerfectSeparationError
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

THRESHOLD = 0.5
PROB_EPS = 1e-15


def _safe_div(num: float, den: float) -> float:
    return float(num / den) if den != 0 else np.nan


def compute_fairness_metrics(y_true: np.ndarray, y_pred: np.ndarray, sex_values: np.ndarray) -> dict:
    """
    Fairness between Class 0 (Female) and Class 1 (Male).
    Zero denominators return np.nan (never silently replaced with 0).
    """
    groups = [0, 1]
    stats = {}
    for g in groups:
        mask = (sex_values == g)
        yt = y_true[mask]
        yp = y_pred[mask]
        tp = int(np.sum((yt == 1) & (yp == 1)))
        tn = int(np.sum((yt == 0) & (yp == 0)))
        fp = int(np.sum((yt == 0) & (yp == 1)))
        fn = int(np.sum((yt == 1) & (yp == 0)))
        n_g = tp + tn + fp + fn

        stats[g] = {
            "SR":  _safe_div(tp + fp, n_g),
            "TPR": _safe_div(tp, tp + fn),
            "FPR": _safe_div(fp, fp + tn),
            "PPV": _safe_div(tp, tp + fp),
        }

    sr0, sr1 = stats[0]["SR"], stats[1]["SR"]
    tpr0, tpr1 = stats[0]["TPR"], stats[1]["TPR"]
    fpr0, fpr1 = stats[0]["FPR"], stats[1]["FPR"]
    ppv0, ppv1 = stats[0]["PPV"], stats[1]["PPV"]

    di = np.nan if (np.isnan(sr0) or np.isnan(sr1) or max(sr0, sr1) == 0.0) else float(min(sr0, sr1) / max(sr0, sr1))
    eod = np.nan if (np.isnan(tpr0) or np.isnan(tpr1)) else float(abs(tpr0 - tpr1))
    dpd = np.nan if (np.isnan(sr0) or np.isnan(sr1)) else float(abs(sr0 - sr1))
    eqod = np.nan if any(np.isnan(v) for v in (tpr0, tpr1, fpr0, fpr1)) else float(max(abs(tpr0 - tpr1), abs(fpr0 - fpr1)))
    ppd = np.nan if (np.isnan(ppv0) or np.isnan(ppv1)) else float(abs(ppv0 - ppv1))

    return {"DI": di, "EOD": eod, "DPD": dpd, "EqOD": eqod, "PPD": ppd}


def compute_discrimination_metrics(y_true: np.ndarray, y_pred: np.ndarray, y_prob: np.ndarray) -> dict:
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))

    return {
        "AUROC":       float(roc_auc_score(y_true, y_prob)) if len(np.unique(y_true)) == 2 else np.nan,
        "Sensitivity": _safe_div(tp, tp + fn),
        "Specificity": _safe_div(tn, tn + fp),
        "PPV":         _safe_div(tp, tp + fp),
        "NPV":         _safe_div(tn, tn + fn),
    }


def compute_calibration_metrics(y_true: np.ndarray, y_prob: np.ndarray) -> dict:
    p = np.clip(np.asarray(y_prob, dtype=np.float64), PROB_EPS, 1.0 - PROB_EPS)
    y = np.asarray(y_true, dtype=np.float64)
    n = len(y)

    # 1. ECE: 15 equal-width bins across [0, 1], final bin closed at 1.0
    bin_edges = np.linspace(0.0, 1.0, 16)
    ece = 0.0
    for b in range(15):
        mask = (p >= bin_edges[b]) & (p < bin_edges[b + 1]) if b < 14 else (p >= bin_edges[b]) & (p <= bin_edges[b + 1])
        n_b = int(np.sum(mask))
        if n_b > 0:
            ece += (n_b / n) * abs(float(np.mean(y[mask])) - float(np.mean(p[mask])))

    # 2. NLL
    nll = float(-np.mean(y * np.log(p) + (1.0 - y) * np.log(1.0 - p)))

    # 3. Brier Score
    brier = float(np.mean((p - y) ** 2))

    # 4-5. Logistic calibration regression: logit(P(Y=1)) = intercept + slope * logit(p)
    cal_slope, cal_intercept = np.nan, np.nan
    logit_p = np.log(p / (1.0 - p))
    if len(np.unique(y)) == 2 and np.std(logit_p) > 1e-12:
        try:
            X_cal = sm.add_constant(logit_p, has_constant="add")
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                res = sm.Logit(y, X_cal).fit(disp=False, maxiter=200)
            if bool(res.mle_retvals.get("converged", True)) and np.all(np.isfinite(res.params)) and len(res.params) == 2:
                cal_intercept = float(res.params[0])
                cal_slope = float(res.params[1])
        except (PerfectSeparationError, np.linalg.LinAlgError, ValueError, RuntimeError):
            cal_slope, cal_intercept = np.nan, np.nan

    return {
        "ECE": float(ece),
        "NLL": nll,
        "Brier": brier,
        "Cal_Slope": cal_slope,
        "Cal_Intercept": cal_intercept,
    }


def compute_robustness_metrics(
    y_true: np.ndarray,
    y_prob_clean: np.ndarray,
    y_pred_clean: np.ndarray,
    y_prob_noisy: np.ndarray,
    y_pred_noisy: np.ndarray,
) -> dict:
    if len(np.unique(y_true)) == 2:
        auroc_deg = float(roc_auc_score(y_true, y_prob_clean)) - float(roc_auc_score(y_true, y_prob_noisy))
    else:
        auroc_deg = np.nan

    acc_deg = float(accuracy_score(y_true, y_pred_clean)) - float(accuracy_score(y_true, y_pred_noisy))

    f1_c = f1_score(y_true, y_pred_clean, pos_label=1, zero_division=np.nan)
    f1_n = f1_score(y_true, y_pred_noisy, pos_label=1, zero_division=np.nan)
    f1_deg = np.nan if (np.isnan(f1_c) or np.isnan(f1_n)) else float(f1_c - f1_n)

    flip_rate = float(np.mean(y_pred_clean != y_pred_noisy))
    mapc = float(np.mean(np.abs(y_prob_clean - y_prob_noisy)))

    return {
        "AUROC_Deg": auroc_deg,
        "Accuracy_Deg": acc_deg,
        "F1_Deg": f1_deg,
        "Flip_Rate": flip_rate,
        "MAPC": mapc,
    }


def evaluate_fold(
    y_true: np.ndarray,
    sex_test: np.ndarray,
    y_prob_clean: np.ndarray,
    y_prob_noisy: np.ndarray,
) -> dict:
    y_true = np.asarray(y_true, dtype=int)
    sex_test = np.asarray(sex_test, dtype=int)
    y_prob_clean = np.asarray(y_prob_clean, dtype=np.float64)
    y_prob_noisy = np.asarray(y_prob_noisy, dtype=np.float64)

    y_pred_clean = (y_prob_clean >= THRESHOLD).astype(int)
    y_pred_noisy = (y_prob_noisy >= THRESHOLD).astype(int)

    out = {}
    out.update(compute_fairness_metrics(y_true, y_pred_clean, sex_test))
    out.update(compute_discrimination_metrics(y_true, y_pred_clean, y_prob_clean))
    out.update(compute_calibration_metrics(y_true, y_prob_clean))
    out.update(compute_robustness_metrics(y_true, y_prob_clean, y_pred_clean, y_prob_noisy, y_pred_noisy))
    return out

In [ ]:
# Cell 5: 5-Fold Stratified CV + Fold-Safe KNN Imputation + Gaussian Noise + TabPFN v2.0 Runner
from sklearn.model_selection import StratifiedKFold
from sklearn.impute import KNNImputer

SEED = 42
N_SPLITS = 5
KNN_NEIGHBORS = 5
NOISE_FRACTION = 0.10
FOLD_CHECKPOINT_CSV = "/content/TabPFN_fold_checkpoint.csv"


def prepare_fold_data(ds_name: str, df: pd.DataFrame, train_idx: np.ndarray, test_idx: np.ndarray, fold_idx: int):
    sex_col = df.columns[0]
    target_col = df.columns[-1]
    feature_cols = list(df.columns[:-1])
    noise_cols = [c for c in feature_cols if c != sex_col]

    X = df[feature_cols]
    y = df[target_col].to_numpy(dtype=int)

    X_train_raw = X.iloc[train_idx].copy()
    X_test_raw = X.iloc[test_idx].copy()
    y_train = y[train_idx]
    y_test = y[test_idx]

    imputer = KNNImputer(n_neighbors=KNN_NEIGHBORS)
    X_train_imp = pd.DataFrame(imputer.fit_transform(X_train_raw), columns=feature_cols, index=X_train_raw.index)
    X_test_clean = pd.DataFrame(imputer.transform(X_test_raw), columns=feature_cols, index=X_test_raw.index)

    X_train_imp[sex_col] = np.round(X_train_imp[sex_col]).astype(int)
    X_test_clean[sex_col] = np.round(X_test_clean[sex_col]).astype(int)
    sex_test = X_test_clean[sex_col].to_numpy(dtype=int)

    ds_hash = sum((i + 1) * ord(ch) for i, ch in enumerate(ds_name))
    fold_rng = np.random.default_rng(SEED * 10000 + ds_hash * 100 + fold_idx)

    X_test_noisy = X_test_clean.copy()
    for col in noise_cols:
        sd_train = float(X_train_imp[col].std(ddof=1))
        if np.isfinite(sd_train) and sd_train > 0.0:
            eps = fold_rng.normal(loc=0.0, scale=1.0, size=len(X_test_noisy))
            X_test_noisy[col] = X_test_clean[col].to_numpy() + (NOISE_FRACTION * sd_train) * eps

    assert np.array_equal(X_test_noisy[sex_col].to_numpy(), X_test_clean[sex_col].to_numpy())
    return X_train_imp, y_train, X_test_clean, X_test_noisy, y_test, sex_test


def build_tabpfn_v2(seed: int = SEED):
    kwargs = {
        "model_path": LOCAL_CKPT_PATH,     # Local file: zero HF network calls
        "device": DEVICE,
        "ignore_pretraining_limits": True,
        "random_state": seed,
    }
    if "show_progress_bar" in _tabpfn_init_params:
        kwargs["show_progress_bar"] = False
    if "fit_mode" in _tabpfn_init_params:
        kwargs["fit_mode"] = "fit_with_cache"  # Cache train context once for both clean & noisy test
    return TabPFNClassifier.create_default_for_version(ModelVersion.V2, **kwargs)


def get_class1_proba(clf, probs: np.ndarray) -> np.ndarray:
    probs = np.asarray(probs, dtype=np.float64)
    return probs[:, list(clf.classes_).index(1)]


CSV_COLUMNS = [
    "Dataset", "Train_N", "Test_N", "Num_Feats", "Domain",
    "DI_mean", "DI_sd",
    "EOD_mean", "EOD_sd",
    "DPD_mean", "DPD_sd",
    "EqOD_mean", "EqOD_sd",
    "PPD_mean", "PPD_sd",
    "AUROC_mean", "AUROC_sd",
    "Sensitivity_mean", "Sensitivity_sd",
    "Specificity_mean", "Specificity_sd",
    "PPV_mean", "PPV_sd",
    "NPV_mean", "NPV_sd",
    "ECE_mean", "ECE_sd",
    "NLL_mean", "NLL_sd",
    "Brier_mean", "Brier_sd",
    "Cal_Slope_mean", "Cal_Slope_sd",
    "Cal_Intercept_mean", "Cal_Intercept_sd",
    "AUROC_Deg_mean", "AUROC_Deg_sd",
    "Accuracy_Deg_mean", "Accuracy_Deg_sd",
    "F1_Deg_mean", "F1_Deg_sd",
    "Flip_Rate_mean", "Flip_Rate_sd",
    "MAPC_mean", "MAPC_sd",
]

METRIC_KEYS = [
    "DI", "EOD", "DPD", "EqOD", "PPD",
    "AUROC", "Sensitivity", "Specificity", "PPV", "NPV",
    "ECE", "NLL", "Brier", "Cal_Slope", "Cal_Intercept",
    "AUROC_Deg", "Accuracy_Deg", "F1_Deg", "Flip_Rate", "MAPC",
]

# Resume from checkpoint if present
if os.path.exists(FOLD_CHECKPOINT_CSV):
    ckpt_df = pd.read_csv(FOLD_CHECKPOINT_CSV)
    completed = set(zip(ckpt_df["Dataset"], ckpt_df["Fold"]))
else:
    ckpt_df = pd.DataFrame()
    completed = set()

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
total_folds = len(DATASET_CONFIGS) * N_SPLITS

with tqdm(
    total=total_folds,
    initial=len(completed),
    desc="TabPFN v2.0 (5-Fold CV)",
    bar_format="{l_bar}{bar}| {percentage:3.0f}% [{elapsed}<{remaining}, {postfix}]",
    leave=True,
) as pbar:
    for cfg in DATASET_CONFIGS:
        ds_name = cfg["Dataset"]
        df = LOADED_DATASETS[ds_name]
        y_all = df[df.columns[-1]].to_numpy(dtype=int)

        for fold_idx, (train_idx, test_idx) in enumerate(skf.split(df.iloc[:, :-1], y_all), start=1):
            if (ds_name, fold_idx) in completed:
                continue

            pbar.set_postfix_str(f"{ds_name} | Fold {fold_idx}/{N_SPLITS}")
            X_tr, y_tr, X_te_clean, X_te_noisy, y_te, sex_te = prepare_fold_data(
                ds_name, df, train_idx, test_idx, fold_idx
            )

            try:
                clf = build_tabpfn_v2(seed=SEED)
                clf.fit(X_tr, y_tr)
                p_clean = get_class1_proba(clf, clf.predict_proba(X_te_clean))
                p_noisy = get_class1_proba(clf, clf.predict_proba(X_te_noisy))
            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    if DEVICE == "cuda":
                        torch.cuda.empty_cache()
                    # Fallback to low_memory mode for very wide datasets (e.g. Arrhythmia)
                    clf = TabPFNClassifier.create_default_for_version(
                        ModelVersion.V2,
                        model_path=LOCAL_CKPT_PATH,
                        device=DEVICE,
                        ignore_pretraining_limits=True,
                        random_state=SEED,
                        fit_mode="low_memory",
                        **({"show_progress_bar": False} if "show_progress_bar" in _tabpfn_init_params else {}),
                    )
                    clf.fit(X_tr, y_tr)
                    p_clean = get_class1_proba(clf, clf.predict_proba(X_te_clean))
                    p_noisy = get_class1_proba(clf, clf.predict_proba(X_te_noisy))
                else:
                    raise

            fold_res = evaluate_fold(y_te, sex_te, p_clean, p_noisy)
            fold_row = {
                "Dataset": ds_name,
                "Fold": fold_idx,
                "Train_N": len(X_tr),
                "Test_N": len(X_te_clean),
                **fold_res,
            }
            ckpt_df = pd.concat([ckpt_df, pd.DataFrame([fold_row])], ignore_index=True)
            ckpt_df.to_csv(FOLD_CHECKPOINT_CSV, index=False)
            completed.add((ds_name, fold_idx))

            del clf
            gc.collect()
            if DEVICE == "cuda":
                torch.cuda.empty_cache()

            pbar.update(1)

# Aggregate per-dataset Mean & Sample SD (ddof=1) into final TabPFN_Results.csv
results_rows = []
for cfg in DATASET_CONFIGS:
    ds_name = cfg["Dataset"]
    df = LOADED_DATASETS[ds_name]
    sub = ckpt_df[ckpt_df["Dataset"] == ds_name]

    mean_train_n = float(sub["Train_N"].mean())
    mean_test_n = float(sub["Test_N"].mean())
    row = {
        "Dataset": ds_name,
        "Train_N": int(mean_train_n) if mean_train_n.is_integer() else mean_train_n,
        "Test_N": int(mean_test_n) if mean_test_n.is_integer() else mean_test_n,
        "Num_Feats": int(df.shape[1] - 1),
        "Domain": cfg["Domain"],
    }

    for m in METRIC_KEYS:
        valid = sub[m].dropna().to_numpy(dtype=np.float64)
        if len(valid) == 0:
            row[f"{m}_mean"] = np.nan
            row[f"{m}_sd"] = np.nan
        else:
            row[f"{m}_mean"] = float(np.mean(valid))
            row[f"{m}_sd"] = float(np.std(valid, ddof=1)) if len(valid) > 1 else np.nan

    results_rows.append(row)

tabpfn_results_df = pd.DataFrame(results_rows)[CSV_COLUMNS]
OUTPUT_CSV = "/content/TabPFN_Results.csv"
tabpfn_results_df.to_csv(OUTPUT_CSV, index=False, na_rep="N/A")
print(f"\nSaved Borsha's deliverable to: {OUTPUT_CSV}")

TabPFN v2.0 (5-Fold CV):   0%|          |   0% [00:00<?, ]


Saved Borsha's deliverable to: /content/TabPFN_Results.csv


In [ ]:
# Cell 6: Verify exact CSV specification and download TabPFN_Results.csv
df_check = pd.read_csv("/content/TabPFN_Results.csv")

assert list(df_check.columns) == CSV_COLUMNS, "Column order mismatch!"
assert len(df_check) == 10, f"Expected 10 dataset rows, got {len(df_check)}"
print(f"Verification passed: {df_check.shape[0]} rows x {df_check.shape[1]} columns in exact required order.")

display(df_check)

# Trigger browser download in Google Colab
try:
    from google.colab import files
    files.download("/content/TabPFN_Results.csv")
except Exception:
    print("File is ready at /content/TabPFN_Results.csv")

Verification passed: 10 rows x 45 columns in exact required order.


,Dataset,Train_N,Test_N,Num_Feats,Domain,DI_mean,DI_sd,EOD_mean,EOD_sd,DPD_mean,...,AUROC_Deg_mean,AUROC_Deg_sd,Accuracy_Deg_mean,Accuracy_Deg_sd,F1_Deg_mean,F1_Deg_sd,Flip_Rate_mean,Flip_Rate_sd,MAPC_mean,MAPC_sd
0,COPD,307.2,76.8,14,Pulmonology / Respiratory Medicine,0.330292,0.109304,0.365126,0.258199,0.462516,...,0.018283,0.024752,3.653452e-02,0.023496,0.043445,0.023319,0.104204,0.018649,0.103725,0.012430
1,Alzheimers,560.0,140.0,8,Neurology / Neurodegenerative Disease,0.916045,0.051285,0.075812,0.058700,0.044806,...,0.005878,0.003987,1.000000e-02,0.008144,0.012412,0.009446,0.052857,0.008144,0.047027,0.008903
2,Arrhythmia,331.2,82.8,262,Cardiology,0.612025,0.175962,0.095438,0.026655,0.247201,...,0.007935,0.023676,1.615339e-01,0.101955,0.079568,0.067636,0.306582,0.112423,0.210495,0.034665
3,Autism,299.2,74.8,19,Neurodevelopmental / Psychiatry,0.864218,0.108950,0.000000,0.000000,0.075502,...,0.000569,0.000595,2.670270e-02,0.016293,0.027796,0.017817,0.026703,0.016293,0.154577,0.017778
4,Diabetes,560.0,140.0,8,Endocrinology / Metabolic Disease,0.922351,0.130877,0.058001,0.048653,0.047124,...,0.009306,0.003537,1.000000e-02,0.010833,0.010907,0.010884,0.061429,0.020578,0.045563,0.005665
5,Stroke,398.4,99.6,10,Neurology / Cerebrovascular Disease,0.832391,0.092058,0.137179,0.054133,0.107499,...,0.001386,0.014872,6.020202e-03,0.005496,0.001736,0.007973,0.034222,0.023281,0.046141,0.009780
6,Heart,560.0,140.0,14,Cardiology,0.272975,0.124641,0.256430,0.248102,0.426981,...,0.007265,0.005087,2.220446e-17,0.011294,-0.001883,0.011269,0.034286,0.011737,0.048501,0.005343
7,Hepatitis,51.2,12.8,19,Hepatology / Infectious Disease,0.000000,0.000000,NaN,NaN,0.555556,...,0.003968,0.027205,3.076923e-02,0.042133,0.035325,0.048405,0.030769,0.042133,0.043008,0.010143
8,Liver,560.0,140.0,10,Hepatology / Gastroenterology,0.801154,0.160532,0.080300,0.046927,0.106631,...,0.135347,0.017082,1.214286e-01,0.027199,0.124366,0.030125,0.172857,0.027386,0.230643,0.006297
9,Thyroid,560.0,140.0,10,Endocrinology,0.853803,0.035615,0.081927,0.062400,0.073523,...,0.145469,0.032512,1.342857e-01,0.025951,0.133575,0.024073,0.182857,0.013924,0.224636,0.020279


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>